# Complete Pipeline: Reproduction of Table 1 (ResNet-18 @ CIFAR-10)

This notebook executes the following steps:

1. **Setup**: Prepares folders and downloads CIFAR-10 + ResNet-18.
2. **Covariance**: Calculates the functional covariance matrix for all layers.
3. **Extraction**: Separates weights and Cholesky factors ($\Sigma^{1/2}$) per layer.
4. **Decomposition**: Executes CP-ALS-Sigma using the MINRES iterative solver.
5. **Report**: Generates the final reconstruction error table.

# Cells to be run in Colab

In [ ]:
!pip install torchinfo nvidia-ml-py3

  Preparing metadata (setup.py) ... done
  Created wheel for nvidia-ml-py3: filename=nvidia_ml_py3-7.352.0-py3-none-any.whl size=19172 sha256=db6c786f3350d1ef2e6f52acc6393cb6fa8bacd8d78ca8b52db85678247daca4
  Stored in directory: /root/.cache/pip/wheels/6e/65/79/33dee66cba26e8204801916dfee7481bccfd22905ebb841fe5
Successfully built nvidia-ml-py3


In [ ]:
!git clone https://github.com/weslleylc/cpd_project_sigma.git

Cloning into 'cpd_project_sigma'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 17 (delta 5), reused 17 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 23.42 KiB | 7.81 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [ ]:
import sys
sys.path.append('cpd_project_sigma')

# Completed cells to run in Colab

In [ ]:
# =============================================================================
# 0. DEPENDENCIES AND CONFIGURATIONS
# =============================================================================
import torch
import torchvision
import os
import math
import random
import numpy as np
import cupy as cp
from tqdm import tqdm
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from torchinfo import summary
from torch.utils.dlpack import from_dlpack, to_dlpack
from cupyx.scipy.sparse.linalg import minres, LinearOperator
from pathlib import Path

# Global Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = os.getcwd()
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Required folders
for folder in ['cifar10_data/images', 'outputs', 'weights']:
    os.makedirs(folder, exist_ok=True)

## 1. Data and Model Setup

We download the pre-trained model and prepare the CIFAR-10 calibration images.

In [ ]:
# --- Download CIFAR-10 and Image Saving ---
print("Downloading CIFAR-10...")
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
num_images = len(dataset)

with open('dataset_files.txt', 'w') as f_files, open('dataset_labels.txt', 'w') as f_labels:
    for i in tqdm(range(num_images), desc="Saving images"):
        img, label = dataset[i]
        img_name = f'img_{i}.png'
        img_path = os.path.join('cifar10_data/images', img_name)
        img.save(img_path)
        f_files.write(f'{img_name}\n')
        f_labels.write(f'{label}\n')

# --- ResNet-18 Model ---
print("Loading ResNet-18...")
model_full = torchvision.models.resnet18(weights="IMAGENET1K_V1")
torch.save(model_full, 'weights/resnet18_full.pt')

100%|██████████| 170M/170M [00:06<00:00, 25.7MB/s]
Saving images: 100%|██████████| 50000/50000 [00:06<00:00, 7519.49it/s]


Loading ResNet-18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 417MB/s]


In [ ]:
# --- Transformation Definition ---
from torchvision import transforms
def get_transform():
    return transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
my_transform = get_transform()

## 2. Covariance Calculation ($\Sigma$)

Integration of the logic from your `Compute_Covariance.py` script to generate the Sigmas dictionary.

In [ ]:
# Feature extraction and Sigma calculation functions...
# (Here we include the simplified logic from compute_covariance)

def run_covariance_step():
    from cpd_project_sigma.Compute_Covariance import _main
    import argparse

    args = argparse.Namespace(
        model='weights/resnet18_full.pt',
        transf=None, # We pass the transform object directly if needed, or save it
        root='cifar10_data/images',
        dataset_files='dataset_files.txt',
        dataset_labels='dataset_labels.txt',
        batch_size=100,
        batch_size_mean=None,
        batch_size_product=None,
        bias=False,
        output='outputs/sigma_resnet18',
        workers=4
    )

    # We save the transform for the script to read (as your _main expects a path)
    torch.save(my_transform, 'my_transform.pt')
    args.transf = 'my_transform.pt'

    print("\nStarting Functional Covariance calculation...")
    _main(args)

run_covariance_step()


Starting Functional Covariance calculation...
Started at: 2026-03-26 16:12:00
Batch size when computing E(X_i)E(Xj): 100
Batch size when computing E(X_i Xj): 100
Using GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Using 4 workers.
Is CUDA available: True
Computing E(X_i)E(Xj)...
Computing the mean of the input of all the conv2d layers...
Start get_mean_pre_conv_input_full
{'x': 'conv1', 'maxpool': 'layer1.0.conv1', 'layer1.0.relu': 'layer1.0.conv2', 'layer1.0.relu_1': 'layer1.1.conv1', 'layer1.1.relu': 'layer1.1.conv2', 'layer1.1.relu_1': 'layer2.0.conv1', 'layer2.0.relu': 'layer2.0.conv2', 'layer2.0.relu_1': 'layer2.1.conv1', 'layer2.1.relu': 'layer2.1.conv2', 'layer2.1.relu_1': 'layer3.0.conv1', 'layer3.0.relu': 'layer3.0.conv2', 'layer3.0.relu_1': 'layer3.1.conv1', 'layer3.1.relu': 'layer3.1.conv2', 'layer3.1.relu_1': 'layer4.0.conv1', 'layer4.0.relu': 'layer4.0.conv2', 'layer4.0.relu_1': 'layer4.1.conv1', 'layer4.1.relu': 'layer4.1.conv2'}
{'conv1', 'layer1.0.conv2', 'layer3.0

  0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  super().__init__(loader)
100%|██████████| 500/500 [00:01<00:00, 327.27it/s]


Full mean RAM use: 1.896189952 GB RAM
GPU memory: 1.03e+02 GB VRAM GPU memory allocated: 9.94e+01 GB VRAM GPU memory free: 3.25e+00 GB VRAM
Full mean are in cuda:0
Computing the product of the mean of the input of all the conv2d layers...
Computing the product of the mean of the input of conv1...
Computing the product of the mean of the input of layer1.0.conv2...
Computing the product of the mean of the input of layer3.0.conv1...
Computing the product of the mean of the input of layer3.1.conv2...
Computing the product of the mean of the input of layer3.0.conv2...
Computing the product of the mean of the input of layer1.1.conv1...
Computing the product of the mean of the input of layer1.0.conv1...
Computing the product of the mean of the input of layer4.1.conv1...
Computing the product of the mean of the input of layer3.1.conv1...
Computing the product of the mean of the input of layer2.0.conv1...
Computing the product of the mean of the input of layer2.1.conv2...
Computing the product 

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  super().__init__(loader)


input_size (100, 3, 32, 32)
Model infos dict_keys(['conv1', 'layer1.0.conv1', 'layer1.0.conv2', 'layer1.1.conv1', 'layer1.1.conv2', 'layer2.0.conv1', 'layer2.0.conv2', 'layer2.1.conv1', 'layer2.1.conv2', 'layer3.0.conv1', 'layer3.0.conv2', 'layer3.1.conv1', 'layer3.1.conv2', 'layer4.0.conv1', 'layer4.0.conv2', 'layer4.1.conv1', 'layer4.1.conv2'])
(100, 3, 7, 15, 3, 32, 32)
(100, 64, 3, 7, 64, 8, 8)
(100, 128, 3, 7, 128, 4, 4)
(100, 256, 3, 7, 256, 2, 2)
(100, 256, 3, 7, 256, 2, 2)
(100, 64, 3, 7, 64, 8, 8)
(100, 64, 3, 7, 64, 8, 8)
(100, 512, 3, 7, 512, 1, 1)
(100, 256, 3, 7, 256, 2, 2)
(100, 64, 3, 7, 64, 8, 8)
(100, 128, 3, 7, 128, 4, 4)
(100, 512, 3, 7, 512, 1, 1)
(100, 512, 3, 7, 512, 1, 1)
(100, 256, 3, 7, 256, 2, 2)
(100, 128, 3, 7, 128, 4, 4)
(100, 128, 3, 7, 128, 4, 4)
(100, 64, 3, 7, 64, 8, 8)


  0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  super().__init__(loader)
100%|██████████| 500/500 [00:19<00:00, 25.39it/s]


Full product RAM use: 1.90490624 GB RAM
GPU memory: 1.03e+02 GB VRAM GPU memory allocated: 9.94e+01 GB VRAM GPU memory free: 3.25e+00 GB VRAM
Full product is in cuda:0
Reshaping the mean of the product of the input of all the conv2d layers...
Mean product is in cuda:0
Computing covariance...
Time to compute the covariance: 21.45s
Saving the covariance...
Computing the Cholesky decomposition of the covariances...
The tensor of layer layer3.1.conv2 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer3.1.conv2.
The tensor of layer layer1.0.conv1 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer1.0.conv1.
The tensor of layer layer4.1.conv1 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer4.1.conv1.
The tensor of layer layer4.1.conv2 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer4.1.conv2.
The tensor of layer layer4.0.conv2 is not positive definite.
Added 1e-09

## 3. Extraction of Weights and Sigmas per Layer

We prepare individual files for the layers in Table 1.

In [ ]:
LAYERS_MAP = {
    'layer1.0.conv2': 134,
    'layer2.0.conv1': 140,
    'layer3.0.conv1': 291,
    'layer3.1.conv1': 520,
    'layer4.1.conv1': 1023,
    'layer4.1.conv2': 1167
}

def extract_layers():
    print("\nExtracting weights and Cholesky factors...")
    sigmas_dict = torch.load('outputs/sigma_resnet18_cholesky_covariance.pt', weights_only=False)
    model = torch.load('weights/resnet18_full.pt', weights_only=False)
    modules = dict(model.named_modules())

    for layer_name in LAYERS_MAP.keys():
        # Save Sigma
        torch.save(sigmas_dict[layer_name], f'outputs/sigma_resnet18_{layer_name}.pt')

        # Save formatted Weights [Out, In, H*W]
        target = modules[layer_name]
        w_raw = target.weight.data
        out_c, in_c, h, w = w_raw.shape
        w_fmt = w_raw.reshape(out_c, in_c, h*w).contiguous()
        torch.save(w_fmt, f'weights/{layer_name.replace(".", "_")}.pt')
        print(f"Layer {layer_name} extracted.")

extract_layers()


Extracting weights and Cholesky factors...
Layer layer1.0.conv2 extracted.
Layer layer2.0.conv1 extracted.
Layer layer3.0.conv1 extracted.
Layer layer3.1.conv1 extracted.
Layer layer4.1.conv1 extracted.
Layer layer4.1.conv2 extracted.


## 4. CP-ALS-Sigma Decomposition

Implementation of the algorithm using the extracted factors and the **MINRES** solver.

In [ ]:
# DLPack utility functions
def torch_to_cupy(t): return cp.from_dlpack(t)
def cupy_to_torch(c): return from_dlpack(c)

def run_all_decompositions():
    results = {}
    from cpd_project_sigma.CP_ALS_Sigma import cp_als_sigma

    print("\nStarting CP-ALS-Sigma Decompositions...")
    for layer_name, rank in LAYERS_MAP.items():
        print(f"\n>>> Processing {layer_name} (Rank {rank})")

        W = torch.load(f'weights/{layer_name.replace(".", "_")}.pt', weights_only=False)
        S_half = torch.load(f'outputs/sigma_resnet18_{layer_name}.pt', weights_only=False)

        factors = cp_als_sigma(W, rank, S_half, n_iter_max=100, verbose=1)

        save_path = f"outputs/cp_factors_{layer_name.replace('.', '_')}_rank{rank}.pt"
        torch.save(factors, save_path)

        # Store paths for the final calculation
        results[layer_name] = {
            'weights': f'weights/{layer_name.replace(".", "_")}.pt',
            'factors': save_path,
            'sigma': f'outputs/sigma_resnet18_{layer_name}.pt',
            'rank': rank
        }
    return results

layer_files = run_all_decompositions()


Starting CP-ALS-Sigma Decompositions...

>>> Processing layer1.0.conv2 (Rank 134)
Iter 1, Erro Sigma: 0.241688, Erro Recon: 5.281182, Erro Recon: 0.608430,Redução: 9.0128e-01
Iter 2, Erro Sigma: 0.119393, Erro Recon: 4.094640, Erro Recon: 0.471732,Redução: 1.2229e-01
Iter 3, Erro Sigma: 0.084765, Erro Recon: 3.731949, Erro Recon: 0.429947,Redução: 3.4628e-02
Iter 4, Erro Sigma: 0.069973, Erro Recon: 3.546593, Erro Recon: 0.408593,Redução: 1.4793e-02
Iter 5, Erro Sigma: 0.060457, Erro Recon: 3.429405, Erro Recon: 0.395092,Redução: 9.5155e-03
Iter 6, Erro Sigma: 0.054111, Erro Recon: 3.346530, Erro Recon: 0.385544,Redução: 6.3464e-03
Iter 7, Erro Sigma: 0.049808, Erro Recon: 3.284388, Erro Recon: 0.378385,Redução: 4.3023e-03
Iter 8, Erro Sigma: 0.046525, Erro Recon: 3.235683, Erro Recon: 0.372774,Redução: 3.2836e-03
Iter 9, Erro Sigma: 0.043826, Erro Recon: 3.196102, Erro Recon: 0.368214,Redução: 2.6987e-03
Iter 10, Erro Sigma: 0.041617, Erro Recon: 3.163205, Erro Recon: 0.364424,Reduçã

## 5. Results Table (Reproduction of Table 1)

Final calculation of the relative error in the functional $\Sigma$ norm.

In [ ]:
def calculate_table():
    print(f"\n{'Layer':<15} | {'Rank':<5} | {'Sigma Error':<12}")
    print("-" * 40)

    # Reference values from your prompt (original Table 1)
    ref_table = {
        'layer1.0.conv2': 0.053, 'layer2.0.conv1': 0.118,
        'layer3.0.conv1': 0.094, 'layer3.1.conv1': 0.070,
        'layer4.1.conv1': 0.063, 'layer4.1.conv2': 0.051
    }

    for name, info in layer_files.items():
        K = torch.load(info['weights'], weights_only=False).to(DEVICE)
        factors = torch.load(info['factors'], weights_only=False)
        S_half = torch.load(info['sigma'], weights_only=False).to(DEVICE)

        # Reconstruction
        u_t, u_s, u_h, u_w = [f.to(DEVICE) for f in factors]
        K_hat = torch.einsum('tr,sr,hr,wr->tshw', u_t, u_s, u_h, u_w)

        # Unfolding and Error
        out_c = K.shape[0]
        K_flat = K.reshape(out_c, -1)
        K_hat_flat = K_hat.reshape(out_c, -1)

        num = torch.norm(torch.matmul(K_flat - K_hat_flat, S_half))
        den = torch.norm(torch.matmul(K_flat, S_half))

        error = (num / den).item()

        print(f"{name:<15} | {info['rank']:<5} | {error:<12.3f} (Ref: {ref_table[name]})")

calculate_table()


Layer           | Rank  | Sigma Error 
----------------------------------------
layer1.0.conv2  | 134   | 0.087        (Ref: 0.053)
layer2.0.conv1  | 140   | 0.137        (Ref: 0.118)
layer3.0.conv1  | 291   | 0.125        (Ref: 0.094)
layer3.1.conv1  | 520   | 0.140        (Ref: 0.07)
layer4.1.conv1  | 1023  | 0.479        (Ref: 0.063)
layer4.1.conv2  | 1167  | 0.331        (Ref: 0.051)
